# ログ解析ユーティリティ集
# Log Analysis Toolkit

フライトログの解析に使える便利な関数とプロットのコレクション。

A collection of useful functions and plots for flight log analysis.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from stampfly_edu import load_flight_log, load_sample_data, plot_trajectory, compare_logs, plot_step_response

print("Toolkit ready! / ツールキット準備完了！")

## 1. ログ読み込み / Load Log

In [ ]:
# Load your log file or sample data
# ログファイルまたはサンプルデータを読み込み

# log = load_flight_log("path/to/your/log.csv")
log = load_sample_data("hover")

print(f"Shape: {log.shape}")
print(f"Columns: {list(log.columns)}")
print(f"Duration: {log['time'].iloc[-1]:.1f} s")
log.describe()

## 2. 全チャンネルプロット / Plot All Channels

In [ ]:
# Plot all numeric columns
# 全数値列をプロット
numeric_cols = log.select_dtypes(include=[np.number]).columns.tolist()
numeric_cols = [c for c in numeric_cols if c != 'time']

n = len(numeric_cols)
fig, axes = plt.subplots(n, 1, figsize=(12, 2.5*n), sharex=True)
if n == 1:
    axes = [axes]

for ax, col in zip(axes, numeric_cols):
    ax.plot(log['time'], log[col], linewidth=0.5)
    ax.set_ylabel(col, fontsize=8)
    ax.grid(True, alpha=0.3)

axes[-1].set_xlabel('Time (s)')
fig.suptitle('All Channels / 全チャンネル', fontsize=14)
fig.tight_layout()
plt.show()

## 3. FFT 周波数分析 / FFT Frequency Analysis

In [ ]:
def plot_fft(data, dt, title="FFT"):
    """Plot single-sided FFT spectrum."""
    n = len(data)
    freq = np.fft.rfftfreq(n, dt)
    spectrum = np.abs(np.fft.rfft(data - np.mean(data))) * 2.0 / n
    
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.semilogy(freq[1:], spectrum[1:])  # Skip DC
    ax.set_xlabel('Frequency (Hz) / 周波数')
    ax.set_ylabel('Amplitude / 振幅')
    ax.set_title(title)
    ax.grid(True, alpha=0.3, which='both')
    plt.tight_layout()
    return fig

# Example: FFT of altitude data
dt = log['time'].iloc[1] - log['time'].iloc[0]
if 'z' in log.columns:
    plot_fft(log['z'].values, dt, title='Altitude FFT / 高度 FFT')
    plt.show()

## 4. 統計サマリー / Statistical Summary

In [ ]:
def flight_summary(log):
    """Generate flight summary statistics."""
    summary = {}
    summary['duration_s'] = log['time'].iloc[-1] - log['time'].iloc[0]
    summary['sample_rate_hz'] = 1.0 / (log['time'].iloc[1] - log['time'].iloc[0])
    summary['n_samples'] = len(log)
    
    if 'z' in log.columns:
        summary['alt_mean_m'] = log['z'].mean()
        summary['alt_std_m'] = log['z'].std()
    
    if 'vbat' in log.columns:
        summary['vbat_start_V'] = log['vbat'].iloc[0]
        summary['vbat_end_V'] = log['vbat'].iloc[-1]
    
    return summary

summary = flight_summary(log)
for k, v in summary.items():
    print(f"{k}: {v}")

## 5. ログ比較 / Log Comparison

In [ ]:
# Compare two datasets
# 2つのデータセットを比較
log1 = load_sample_data("hover")
log2 = load_sample_data("altitude_step")

fig = compare_logs(log1, log2, columns=['z'], labels=('Hover', 'Alt Step'))
plt.show()